<a href="https://colab.research.google.com/github/A-ghori/Pytorch-/blob/main/Complete_breast_cancer_neural_network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [2]:
data = pd.read_csv("data.csv")
data.sample(15)

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
548,923169,B,9.683,19.34,61.05,285.7,0.08491,0.05030,0.02337,0.009615,...,25.59,69.10,364.2,0.1199,0.09546,0.09350,0.03846,0.2552,0.07920,NaN
2,84300903,M,19.690,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.127900,...,25.53,152.50,1709.0,0.1444,0.42450,0.45040,0.24300,0.3613,0.08758,NaN
474,911391,B,10.880,15.62,70.41,358.9,0.10070,0.10690,0.05115,0.015710,...,19.35,80.78,433.1,0.1332,0.38980,0.33650,0.07966,0.2581,0.10800,NaN
26,852763,M,14.580,21.53,97.41,644.8,0.10540,0.18680,0.14250,0.087830,...,33.21,122.40,896.9,0.1525,0.66430,0.55390,0.27010,0.4264,0.12750,NaN
198,877486,M,19.180,22.49,127.50,1148.0,0.08523,0.14280,0.11140,0.067720,...,32.06,166.40,1688.0,0.1322,0.56010,0.38650,0.17080,0.3193,0.09221,NaN
245,884437,B,10.480,19.86,66.72,337.7,0.10700,0.05971,0.04831,0.030700,...,29.46,73.68,402.8,0.1515,0.10260,0.11810,0.06736,0.2883,0.07748,NaN
225,88143502,B,14.340,13.47,92.51,641.2,0.09906,0.07624,0.05724,0.046030,...,16.90,110.40,873.2,0.1297,0.15250,0.16320,0.10870,0.3062,0.06072,NaN
498,914769,M,18.490,17.52,121.30,1068.0,0.10120,0.13170,0.14910,0.091830,...,22.88,146.40,1600.0,0.1412,0.30890,0.35330,0.16630,0.2510,0.09445,NaN
484,912600,B,15.730,11.28,102.80,747.2,0.10430,0.12990,0.11910,0.062110,...,14.20,112.50,854.3,0.1541,0.29790,0.40040,0.14520,0.2557,0.08181,NaN
66,859464,B,9.465,21.01,60.11,269.4,0.10440,0.07773,0.02172,0.015040,...,31.56,67.03,330.7,0.1548,0.16640,0.09412,0.06517,0.2878,0.09211,NaN


In [3]:
data.drop(columns=["id", "Unnamed: 32"], inplace=True)

In [4]:
X = data.drop(columns=["diagnosis"])
y = data["diagnosis"]

In [5]:
y

,diagnosis
0,M
1,M
2,M
3,M
4,M
...,...
564,M
565,M
566,M
567,M


In [6]:
X.shape
y.shape

(569,)

In [7]:
X_train,X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)


In [8]:
# Scaling
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [9]:
# Label Encoder
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [10]:
# Numpy array to pytorch tensor

import torch

X_train_tensor = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.float32
).reshape(-1, 1)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.float32
).reshape(-1, 1)


In [11]:
X_train_tensor

tensor([[-1.4408, -0.4353, -1.3621,  ...,  0.9320,  2.0972,  1.8865],
        [ 1.9741,  1.7330,  2.0917,  ...,  2.6989,  1.8912,  2.4978],
        [-1.4000, -1.2496, -1.3452,  ..., -0.9702,  0.5976,  0.0579],
        ...,
        [ 0.0488, -0.5550, -0.0651,  ..., -1.2390, -0.7086, -1.2715],
        [-0.0390,  0.1021, -0.0314,  ...,  1.0500,  0.4343,  1.2134],
        [-0.5486,  0.3133, -0.6035,  ..., -0.6110, -0.3345, -0.8463]])

In [12]:
import torch
import torch.nn as nn

class Model(nn.Module):
  def __init__(self, num_features):

    super().__init__()

    self.loss_fn = torch.nn.BCELoss()
    self.Network = nn.Sequential(
        nn.Linear(num_features, 128),
        nn.ReLU(),
        nn.Linear(128,1),
        nn.Sigmoid(),
    )
  def forward(self,features):
    out = self.Network(features)
    return out

  def loss_func(self,out, y):
    loss = self.loss_fn(out, y)
    return loss


In [13]:
learning_rate = 0.1
epochs = 1000

In [23]:
# create model
model = Model(X_train_tensor.shape[1])

# define optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

for epoch in range(epochs):

# forward pass
  y_pred = model.forward(X_train_tensor)
# loss calc
  loss = model.loss_func(y_pred, y_train_tensor)
# clear gradient
  optimizer.zero_grad()
# backward pass
  loss.backward()

# updating trainable parameters like weights and biases
  optimizer.step()

  # with torch.no_grad():
  #   for param in model.parameters():
  #     param.data -= learning_rate * param.grad

     # Reset gradients
    # model.zero_grad()

  print(f"Epoch: {epoch+1}, Loss: {loss.item():.4f}")

  # with torch.no_grad():
  #   model.linear.weight -= learning_rate * model.linear.weight.grad
  #   model.linear.bias -= learning_rate * model.linear.weight.grad



Epoch: 1, Loss: 0.7042
Epoch: 2, Loss: 0.6119
Epoch: 3, Loss: 0.5426
Epoch: 4, Loss: 0.4887
Epoch: 5, Loss: 0.4455
Epoch: 6, Loss: 0.4100
Epoch: 7, Loss: 0.3805
Epoch: 8, Loss: 0.3554
Epoch: 9, Loss: 0.3339
Epoch: 10, Loss: 0.3154
Epoch: 11, Loss: 0.2991
Epoch: 12, Loss: 0.2849
Epoch: 13, Loss: 0.2723
Epoch: 14, Loss: 0.2610
Epoch: 15, Loss: 0.2509
Epoch: 16, Loss: 0.2418
Epoch: 17, Loss: 0.2336
Epoch: 18, Loss: 0.2262
Epoch: 19, Loss: 0.2193
Epoch: 20, Loss: 0.2131
Epoch: 21, Loss: 0.2073
Epoch: 22, Loss: 0.2020
Epoch: 23, Loss: 0.1971
Epoch: 24, Loss: 0.1925
Epoch: 25, Loss: 0.1882
Epoch: 26, Loss: 0.1843
Epoch: 27, Loss: 0.1805
Epoch: 28, Loss: 0.1770
Epoch: 29, Loss: 0.1737
Epoch: 30, Loss: 0.1706
Epoch: 31, Loss: 0.1676
Epoch: 32, Loss: 0.1648
Epoch: 33, Loss: 0.1622
Epoch: 34, Loss: 0.1596
Epoch: 35, Loss: 0.1572
Epoch: 36, Loss: 0.1549
Epoch: 37, Loss: 0.1527
Epoch: 38, Loss: 0.1506
Epoch: 39, Loss: 0.1486
Epoch: 40, Loss: 0.1467
Epoch: 41, Loss: 0.1448
Epoch: 42, Loss: 0.1431
E

In [24]:
model.eval()

with torch.no_grad():
    y_pred = model(X_test_tensor)

    predicted_labels = (y_pred >= 0.5).float()

    accuracy = (predicted_labels == y_test_tensor).float().mean()

print("Test Accuracy:", accuracy.item() * 100)

Test Accuracy: 97.36841917037964
